# 09 — UAVDT: second dataset for the iso-cost study

**Project:** RAOD++ · **Phase:** second-dataset confirmation (protocol v2.1 §C4.3)

H2 currently rests on VisDrone alone. UAVDT is the right second test: same domain (UAV
imagery) but different scale statistics — the literature notes that scale variation is less
pronounced in UAVDT than in VisDrone. If resolution's advantage on the energy axis is driven
by small objects, the effect should be *weaker* here. That makes this a mechanism test, not
a repetition, and it puts a boundary on the claim.

This notebook:
1. downloads UAVDT from a Kaggle mirror (cocodataset.org and HF's CDN are blocked here),
2. inspects the layout before assuming anything,
3. converts annotations to YOLO format,
4. **splits by video sequence, never by frame** — consecutive frames are near-duplicates,
   so a random frame split would be severe leakage,
5. temporally subsamples to a VisDrone-comparable size,
6. freezes the split with hashes, audits leakage, writes the data card.

**Runtime:** download 20–60 min depending on the network, the rest ~10 min. CPU only.


## [Cell 3] Config

In [1]:
# ===== Cell 3 =====
from pathlib import Path
import shutil

CONFIG = {
    "PROJECT_ROOT": Path.home() / "RAODpp",
    "KAGGLE_SLUG": "shakaibkaggle/uavdt-dataset",
    "FRAME_STRIDE": 5,        # keep every Nth frame; consecutive frames are redundant
    "VAL_SEQ_FRACTION": 0.25, # share of sequences held out for evaluation
    "SEED": 42,
    "TARGET_TRAIN": 6000,     # comparable to VisDrone's 6,471
    "MIN_BOX_PX": 2,
}

ROOT = CONFIG["PROJECT_ROOT"]
DIRS = {k: ROOT / k for k in ["dataset", "model", "output", "fig", "tex", "notebooks"]}
for d in DIRS.values():
    d.mkdir(parents=True, exist_ok=True)
RAW = DIRS["dataset"] / "uavdt_raw"
OUT = DIRS["dataset"] / "uavdt"
RAW.mkdir(parents=True, exist_ok=True)

free_gb = shutil.disk_usage(Path.home()).free / 1e9
print(f"free space: {free_gb:.1f} GB")
assert free_gb > 30, "need >30 GB free"
print("raw ->", RAW)
print("prepared ->", OUT)


free space: 835.1 GB
raw -> /home/aakkaya/RAODpp/dataset/uavdt_raw
prepared -> /home/aakkaya/RAODpp/dataset/uavdt


## [Cell 5] Download
Kaggle is reachable on this network; the COCO servers and the Hugging Face LFS CDN are not.
If credentials are missing, create an API token at kaggle.com → Settings → API and place
`kaggle.json` under `~/.config/kaggle/`.

In [2]:
# ===== Cell 5 =====
import subprocess, sys

zips = list(RAW.glob("*.zip"))
if not zips:
    r = subprocess.run([sys.executable, "-m", "kaggle", "datasets", "download",
                        "-d", CONFIG["KAGGLE_SLUG"], "-p", str(RAW)],
                       capture_output=True, text=True)
    print(r.stdout[-1500:] or r.stderr[-1500:])
    zips = list(RAW.glob("*.zip"))

for z in zips:
    print(f"{z.name}: {z.stat().st_size/1e9:.2f} GB")
assert zips, "download produced no zip — check kaggle credentials and network"


Authentication required to call the Kaggle API.

First, you will need a Kaggle account. You can sign up at
  https://www.kaggle.com/account/login

Recommended: log in with OAuth via a web-based authorization flow.
No token to manage; credentials are cached locally for you.
    kaggle auth login

If you'd rather not use OAuth, generate an API token at
  https://www.kaggle.com/settings/api  (click "Generate New Token" under "API")
and supply it to the CLI in one of these ways:

  Option A: Environment variable
    export KAGGLE_API_TOKEN=xxxxxxxxxxxxxx  # token copied from the settings UI

  Option B: API token file
    Save the token to ~/.kaggle/access_token
Dataset URL: https://www.kaggle.com/datasets/shakaibkaggle/uavdt-dataset
License(s): MIT


uavdt-dataset.zip: 12.09 GB


## [Cell 7] Inspect the archive before extracting
Mirror layouts differ from the official release, so nothing is assumed: the archive is
listed first and the extraction paths are derived from what is actually inside.

In [ ]:
# ===== Cell 7 =====
import zipfile
from collections import Counter

z = zipfile.ZipFile(zips[0])
names = z.namelist()
print("entries:", len(names))
print("\n--- first 15 ---")
for n in names[:15]:
    print("  ", n)

ext = Counter(Path(n).suffix.lower() for n in names)
print("\n--- extensions ---", dict(ext.most_common(8)))

tops = Counter(n.split("/")[0] for n in names if "/" in n)
print("--- top-level dirs ---", dict(tops.most_common(10)))

depth2 = Counter("/".join(n.split("/")[:2]) for n in names if n.count("/") >= 2)
print("\n--- sample second level ---")
for k, v in list(depth2.most_common(10)):
    print(f"   {k}: {v}")


entries: 233460

--- first 15 ---
   LICENSE.md
   README.md
   meta.json
   test/ann/M0203_img000001.jpg.json
   test/ann/M0203_img000002.jpg.json
   test/ann/M0203_img000003.jpg.json
   test/ann/M0203_img000004.jpg.json
   test/ann/M0203_img000005.jpg.json
   test/ann/M0203_img000006.jpg.json
   test/ann/M0203_img000007.jpg.json
   test/ann/M0203_img000008.jpg.json
   test/ann/M0203_img000009.jpg.json
   test/ann/M0203_img000010.jpg.json
   test/ann/M0203_img000011.jpg.json
   test/ann/M0203_img000012.jpg.json

--- extensions --- {'.json': 155639, '.jpg': 77819, '.md': 2}
--- top-level dirs --- {'test': 161028, 'train': 72429}

--- sample second level ---
   test/ann: 53676
   test/img: 53676
   test/meta: 53676
   train/ann: 24143
   train/img: 24143
   train/meta: 24143


: 

## [Cell 9] Extract
Only what is needed. Re-running is cheap because existing files are skipped.

In [ ]:
# ===== Cell 9 =====
import time

marker = RAW / ".extracted"
if not marker.exists():
    t0 = time.time()
    for i, n in enumerate(names, 1):
        z.extract(n, RAW)
        if i % 5000 == 0:
            print(f"   {i}/{len(names)}  {time.time()-t0:.0f}s", flush=True)
    marker.write_text("done")
    print(f"extracted in {(time.time()-t0)/60:.1f} min")
else:
    print("already extracted")

imgs = sorted(RAW.rglob("*.jpg"))
labels = sorted(list(RAW.rglob("*.txt")) + list(RAW.rglob("*.xml")))
print("images:", len(imgs))
print("annotation files:", len(labels))
print("sample image :", imgs[0] if imgs else "none")
print("sample label :", labels[0] if labels else "none")
if labels:
    print("\n--- first label file, first 5 lines ---")
    print("\n".join(open(labels[0]).read().splitlines()[:5]))


## [Cell 11] Identify sequences
UAVDT frames come from ~100 video sequences. The sequence id is the directory or filename
prefix; it is derived here rather than assumed, and the result is printed for inspection.

In [ ]:
# ===== Cell 11 =====
import re
import pandas as pd

def sequence_of(p: Path):
    for part in p.parts[::-1]:
        m = re.match(r"^(M\d{4})", part)      # UAVDT sequences are named M0101, M0202, ...
        if m:
            return m.group(1)
    return p.parent.name

frame = pd.DataFrame({"path": imgs})
frame["seq"] = [sequence_of(p) for p in frame.path]
frame["stem"] = [p.stem for p in frame.path]

print("sequences found:", frame.seq.nunique())
print(frame.seq.value_counts().head(10).to_string())
assert frame.seq.nunique() > 5, ("sequence detection failed — inspect the paths above and "
                                 "adjust sequence_of()")


## [Cell 13] Sequence-level split, then temporal subsampling

Splitting by frame would be severe leakage: consecutive frames of a video are near
identical, so a model would be evaluated on frames it effectively trained on. Sequences are
therefore assigned wholly to train or val, and only then are frames thinned by stride.

In [ ]:
# ===== Cell 13 =====
import numpy as np

rng = np.random.default_rng(CONFIG["SEED"])
seqs = sorted(frame.seq.unique())
rng.shuffle(seqs)
n_val = max(1, int(round(CONFIG["VAL_SEQ_FRACTION"] * len(seqs))))
val_seqs, train_seqs = set(seqs[:n_val]), set(seqs[n_val:])

frame["split"] = np.where(frame.seq.isin(val_seqs), "val", "train")
frame = frame.sort_values(["seq", "stem"]).reset_index(drop=True)
frame["idx_in_seq"] = frame.groupby("seq").cumcount()

stride = CONFIG["FRAME_STRIDE"]
n_train_full = int((frame.split == "train").sum())
if n_train_full / stride > CONFIG["TARGET_TRAIN"]:
    stride = int(np.ceil(n_train_full / CONFIG["TARGET_TRAIN"]))
    print(f"stride raised to {stride} to hit ~{CONFIG['TARGET_TRAIN']} training frames")

keep = frame[frame.idx_in_seq % stride == 0].copy()
print(f"\nsequences: {len(train_seqs)} train / {len(val_seqs)} val")
print(f"frames   : {len(frame):,} total -> {len(keep):,} kept (stride {stride})")
print(keep.split.value_counts().to_string())
assert not (set(keep[keep.split=='train'].seq) & set(keep[keep.split=='val'].seq)), \
    "LEAKAGE: a sequence appears in both splits"
print("\nsequence-level separation: OK")


## [Cell 15] Convert to YOLO format and materialise the split
Annotation format is detected from the files themselves; if it is not recognised the cell
stops rather than writing silently wrong labels.

In [ ]:
# ===== Cell 15 =====
from PIL import Image

CLASSES = ["car", "truck", "bus"]          # UAVDT's three vehicle categories
for split in ["train", "val"]:
    (OUT / f"{split}" / "images").mkdir(parents=True, exist_ok=True)
    (OUT / f"{split}" / "labels").mkdir(parents=True, exist_ok=True)

label_lookup = {}
for lp in labels:
    label_lookup.setdefault(lp.stem, []).append(lp)

def parse_yolo(lp, W, H):
    out = []
    for line in open(lp):
        f = line.split()
        if len(f) < 5:
            continue
        c = int(float(f[0])); cx, cy, bw, bh = map(float, f[1:5])
        if max(cx, cy, bw, bh) > 1.5:                    # absolute -> normalise
            cx, cy, bw, bh = cx / W, cy / H, bw / W, bh / H
        if bw * W < CONFIG["MIN_BOX_PX"] or bh * H < CONFIG["MIN_BOX_PX"]:
            continue
        out.append(f"{c} {cx:.6f} {cy:.6f} {bw:.6f} {bh:.6f}")
    return out

n_written, n_boxes, n_missing = 0, 0, 0
for r in keep.itertuples():
    lps = label_lookup.get(r.stem)
    if not lps:
        n_missing += 1
        continue
    W, H = Image.open(r.path).size
    lines = parse_yolo(lps[0], W, H)
    dst_img = OUT / r.split / "images" / f"{r.seq}_{r.stem}.jpg"
    if not dst_img.exists():
        dst_img.symlink_to(r.path)
    (OUT / r.split / "labels" / f"{r.seq}_{r.stem}.txt").write_text("\n".join(lines))
    n_written += 1; n_boxes += len(lines)

print(f"frames written : {n_written:,}")
print(f"boxes          : {n_boxes:,}")
print(f"frames without an annotation file: {n_missing:,}")
assert n_written > 1000, "too few frames written — check the annotation lookup"


## [Cell 17] Freeze the split and audit

In [ ]:
# ===== Cell 17 =====
import hashlib, json

hashes = {}
for split in ["train", "val"]:
    files = sorted(p.name for p in (OUT / split / "images").glob("*.jpg"))
    txt = "\n".join(files)
    (DIRS["output"] / f"uavdt_{split}_files.txt").write_text(txt)
    hashes[split] = {"n_files": len(files),
                     "sha256": hashlib.sha256(txt.encode()).hexdigest()}
    print(f"{split}: n={len(files)}  sha256={hashes[split]['sha256'][:16]}...")

hashes["stride"] = int(stride)
hashes["val_sequences"] = sorted(val_seqs)
hashes["train_sequences"] = sorted(train_seqs)
json.dump(hashes, open(DIRS["output"] / "uavdt_split_hashes.json", "w"), indent=2)

tr = {p.name for p in (OUT / "train" / "images").glob("*.jpg")}
va = {p.name for p in (OUT / "val" / "images").glob("*.jpg")}
assert not (tr & va), "filename overlap between splits"
assert not (set(s for s in train_seqs) & set(val_seqs)), "sequence overlap"
print("\nleakage audit: PASS (no shared filenames, no shared sequences)")


## [Cell 19] Data YAML and data card

In [ ]:
# ===== Cell 19 =====
import pandas as pd

yaml_path = DIRS["dataset"] / "uavdt_raodpp.yaml"
yaml_path.write_text(
    f"# RAOD++ UAVDT (second dataset) — generated by 09_data_uavdt.ipynb\n"
    f"path: {OUT}\ntrain: train/images\nval: val/images\nnames:\n"
    + "\n".join(f"  {i}: {n}" for i, n in enumerate(CLASSES)) + "\n")
print(yaml_path.read_text())

counts = {}
for split in ["train", "val"]:
    n_b = sum(1 for p in (OUT / split / "labels").glob("*.txt") for _ in open(p))
    counts[split] = n_b

card = pd.DataFrame([
    ["Dataset", "UAVDT (Kaggle mirror)"],
    ["Role in study", "Second dataset, H2 confirmation (protocol v2.1 C4.3)"],
    ["Source", CONFIG["KAGGLE_SLUG"]],
    ["Sequences", f"{len(train_seqs)} train / {len(val_seqs)} val"],
    ["Split rule", "by video sequence; frames are near-duplicates"],
    ["Frame stride", stride],
    ["Train images", hashes["train"]["n_files"]],
    ["Val images", hashes["val"]["n_files"]],
    ["Train boxes", counts["train"]],
    ["Val boxes", counts["val"]],
    ["Classes", ", ".join(CLASSES)],
    ["Frames without annotation", n_missing],
    ["Split hash (train)", hashes["train"]["sha256"][:32] + "..."],
    ["Split hash (val)", hashes["val"]["sha256"][:32] + "..."],
], columns=["field", "value"])
card.to_csv(DIRS["output"] / "datacard_uavdt.csv", index=False)
card


**Report back:** the archive listing, the sequence count, the split sizes and the
leakage audit line. The conversion cell is the one most likely to need adapting, because
mirror annotation formats vary; if it stops, paste the sample label lines printed earlier.

**Next:** the reduced grid on UAVDT — resolution arm n@640/960/1280 and capacity arm
s@640/960, 3 seeds, ~50 h.